# 1. Imports y config

In [1]:
import numpy as np
import pandas as pd
import os

In [2]:
from config import CONFIG

In [3]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
np.random.seed(42)

# 2. Carga de datos

In [4]:
from features import load_data, impute_flag_nans, assign_split, build_target

In [5]:
spain_path = CONFIG["data_path_spain"]
clm_path = CONFIG["data_path_clm"]

spain_df = load_data(spain_path)
clm_df = load_data(clm_path)

Cargados datos desde ../datasets/parquet/LABELED_CURATED_AEMET_KOPPEN.parquet:
    799338 registros de 66 estaciones
Cargados datos desde ../datasets_clm/main/LABELED_KOPPEN_DATA.parquet:
    339698 registros de 62 estaciones


## 2.1. Imputación de flags de extremo a 0

In [6]:
spain_df = impute_flag_nans(spain_df)

Imputado False en 1853 valores perdidos de extreme_tmax
Imputado False en 2040 valores perdidos de extreme_tmin
Imputado False en 36826 valores perdidos de extreme_prec
Imputado False en 11467 valores perdidos de extreme_racha


In [7]:
clm_df = impute_flag_nans(clm_df)

Imputado False en 0 valores perdidos de extreme_tmax
Imputado False en 0 valores perdidos de extreme_tmin
Imputado False en 0 valores perdidos de extreme_prec
Imputado False en 0 valores perdidos de extreme_racha


## 2.2. Asignacion de splits

In [8]:
spain_df = assign_split(spain_df)

        filas  anio_min  anio_max
split                            
train  654982      1990      2018
val     72252      2019      2021
test    72104      2022      2024


In [9]:
clm_df = assign_split(clm_df)

        filas  anio_min  anio_max
split                            
train  203794      2010      2018
val     67952      2019      2021
test    67952      2022      2024


# 3. Construcción del target

In [10]:
spain_df = build_target(spain_df)
clm_df = build_target(clm_df)

In [11]:
tmax_target_cols = [
    'fecha', 'indicativo', 'tmax', 'tmax_threshold', 'extreme_tmax',
    'target_tmax_1', 'target_tmax_3', 'target_tmax_7'
]

In [12]:
spain_df.loc[
    (spain_df['nombre'] == 'ALBACETE BASE AÉREA') 
    & (spain_df['anio'] == 2015) & (spain_df['mes'].isin([7]))
    & ((spain_df['dia'] <= 29) & (spain_df['dia'] >= 21))
, tmax_target_cols]

,fecha,indicativo,tmax,tmax_threshold,extreme_tmax,target_tmax_1,target_tmax_3,target_tmax_7
544670,2015-07-21,8175,36.799999,37.959999,0,0,0,0
544671,2015-07-22,8175,36.599998,38.000000,0,0,0,1
544672,2015-07-23,8175,35.299999,38.000000,0,0,0,1
544673,2015-07-24,8175,37.200001,38.000000,0,0,0,1
544674,2015-07-25,8175,34.400002,38.000000,0,0,0,1
544675,2015-07-26,8175,36.900002,38.200001,0,0,1,1
544676,2015-07-27,8175,38.200001,38.200001,0,0,1,1
544677,2015-07-28,8175,37.799999,38.400002,0,1,1,1
544678,2015-07-29,8175,38.799999,38.480000,1,0,0,0


# 4. *Feature Engineering*

In [13]:
from features import (
    add_dist_to_ex, add_time_vars, 
    add_cont_vars, add_cat_vars, add_ex_today, 
    add_lags, add_ewma, add_rolls, add_diffs,
    add_prec_vars, add_ex_history, add_aux
)

## 4.1. Registros de features

In [14]:
SPAIN_FEATURE_REG = []
CLM_FEATURE_REG = []

## 4.2. Distancia al umbral de extremo

In [15]:
print("ESPAÑA")
spain_df = add_dist_to_ex(spain_df, SPAIN_FEATURE_REG, ["minimal", "curated"])

ESPAÑA
minimal: añadidas distancias a extremos de ['tmax', 'tmin', 'prec', 'racha']
curated: añadidas distancias a extremos de ['tmax', 'tmin', 'prec', 'racha']


In [16]:
print("CLM")
clm_df = add_dist_to_ex(clm_df, CLM_FEATURE_REG, ["minimal", "curated"])

CLM
minimal: añadidas distancias a extremos de ['tmax', 'tmin', 'prec', 'racha']
curated: añadidas distancias a extremos de ['tmax', 'tmin', 'prec', 'racha']


In [17]:
spain_df.loc[
    (spain_df['nombre'] == 'ALBACETE BASE AÉREA') 
    & (spain_df['anio'] == 2015) & (spain_df['mes'].isin([7]))
    & ((spain_df['dia'] <= 29) & (spain_df['dia'] >= 21))
, tmax_target_cols + ["tmax_dist_to_ex"]]

,fecha,indicativo,tmax,tmax_threshold,extreme_tmax,target_tmax_1,target_tmax_3,target_tmax_7,tmax_dist_to_ex
544670,2015-07-21,8175,36.799999,37.959999,0,0,0,0,-1.160000
544671,2015-07-22,8175,36.599998,38.000000,0,0,0,1,-1.400002
544672,2015-07-23,8175,35.299999,38.000000,0,0,0,1,-2.700001
544673,2015-07-24,8175,37.200001,38.000000,0,0,0,1,-0.799999
544674,2015-07-25,8175,34.400002,38.000000,0,0,0,1,-3.599998
544675,2015-07-26,8175,36.900002,38.200001,0,0,1,1,-1.299999
544676,2015-07-27,8175,38.200001,38.200001,0,0,1,1,0.000000
544677,2015-07-28,8175,37.799999,38.400002,0,1,1,1,-0.600002
544678,2015-07-29,8175,38.799999,38.480000,1,0,0,0,0.320000


## 4.3. Características temporales

In [18]:
print("ESPAÑA")
spain_df = add_time_vars(spain_df, SPAIN_FEATURE_REG, ["minimal", "curated"])

ESPAÑA
minimal: añadidas variables temporales
curated: añadidas variables temporales


In [19]:
print("CLM")
clm_df = add_time_vars(clm_df, CLM_FEATURE_REG, ["minimal", "curated"])

CLM
minimal: añadidas variables temporales
curated: añadidas variables temporales


## 4.4. Características continuas y categóricas

In [20]:
print("ESPAÑA")
spain_df = add_cont_vars(spain_df, SPAIN_FEATURE_REG, ["minimal", "curated"])
spain_df = add_cat_vars(spain_df, SPAIN_FEATURE_REG, ["minimal", "curated"])

ESPAÑA
minimal: añadidas ['tmax', 'tmin', 'prec', 'racha', 'tmed', 'hrMedia', 'mslp_max', 'mslp_min', 'mslp_media', 'velmedia', 'altitud', 'latitud', 'longitud'] (continuas)
curated: añadidas ['tmax', 'tmin', 'prec', 'racha', 'tmed', 'hrMedia', 'mslp_max', 'mslp_min', 'mslp_media', 'velmedia', 'altitud', 'latitud', 'longitud'] (continuas)


minimal: añadidas ['koppen_main', 'koppen_class'] (categóricas)
curated: añadidas ['koppen_main', 'koppen_class'] (categóricas)


In [21]:
print("CLM")
clm_df = add_cont_vars(clm_df, CLM_FEATURE_REG, ["minimal", "curated"])
clm_df = add_cat_vars(clm_df, CLM_FEATURE_REG, ["minimal", "curated"])

CLM
minimal: añadidas ['tmax', 'tmin', 'prec', 'racha', 'tmed', 'hrMedia', 'mslp_max', 'mslp_min', 'mslp_media', 'velmedia', 'altitud', 'latitud', 'longitud'] (continuas)
curated: añadidas ['tmax', 'tmin', 'prec', 'racha', 'tmed', 'hrMedia', 'mslp_max', 'mslp_min', 'mslp_media', 'velmedia', 'altitud', 'latitud', 'longitud'] (continuas)
minimal: añadidas ['koppen_main', 'koppen_class'] (categóricas)
curated: añadidas ['koppen_main', 'koppen_class'] (categóricas)


## 4.5. Indicador de extremo

In [22]:
print("ESPAÑA")
spain_df = add_ex_today(spain_df, SPAIN_FEATURE_REG, ["minimal", "curated"])

ESPAÑA
minimal: añadidos indicadores de extremos de ['tmax', 'tmin', 'prec', 'racha']
curated: añadidos indicadores de extremos de ['tmax', 'tmin', 'prec', 'racha']


In [23]:
print("CLM")
clm_df = add_ex_today(clm_df, CLM_FEATURE_REG, ["minimal", "curated"])

CLM
minimal: añadidos indicadores de extremos de ['tmax', 'tmin', 'prec', 'racha']
curated: añadidos indicadores de extremos de ['tmax', 'tmin', 'prec', 'racha']


## 4.6. Lags

In [24]:
print("ESPAÑA")
spain_df = add_lags(spain_df, SPAIN_FEATURE_REG, ["curated"])

ESPAÑA


curated: añadidos lags [1, 3] de ['tmax', 'tmin', 'prec', 'racha', 'mslp_min', 'mslp_max']


In [25]:
print("CLM")
clm_df = add_lags(clm_df, CLM_FEATURE_REG, ["curated"])

CLM
curated: añadidos lags [1, 3] de ['tmax', 'tmin', 'prec', 'racha', 'mslp_min', 'mslp_max']


In [26]:
spain_df.loc[
    (spain_df['nombre'] == 'ALBACETE BASE AÉREA') 
    & (spain_df['anio'] == 2015) & (spain_df['mes'].isin([7]))
    & ((spain_df['dia'] <= 29) & (spain_df['dia'] >= 21))
, tmax_target_cols + ["tmax", "tmax_lag_1", "tmax_lag_3"]]

,fecha,indicativo,tmax,tmax_threshold,extreme_tmax,target_tmax_1,target_tmax_3,target_tmax_7,tmax,tmax_lag_1,tmax_lag_3
544670,2015-07-21,8175,36.799999,37.959999,0,0,0,0,36.799999,37.000000,34.000000
544671,2015-07-22,8175,36.599998,38.000000,0,0,0,1,36.599998,36.799999,34.799999
544672,2015-07-23,8175,35.299999,38.000000,0,0,0,1,35.299999,36.599998,37.000000
544673,2015-07-24,8175,37.200001,38.000000,0,0,0,1,37.200001,35.299999,36.799999
544674,2015-07-25,8175,34.400002,38.000000,0,0,0,1,34.400002,37.200001,36.599998
544675,2015-07-26,8175,36.900002,38.200001,0,0,1,1,36.900002,34.400002,35.299999
544676,2015-07-27,8175,38.200001,38.200001,0,0,1,1,38.200001,36.900002,37.200001
544677,2015-07-28,8175,37.799999,38.400002,0,1,1,1,37.799999,38.200001,34.400002
544678,2015-07-29,8175,38.799999,38.480000,1,0,0,0,38.799999,37.799999,36.900002


## 4.7. Rolls

In [27]:
# spain_df.groupby("indicativo")["tmax"].rolling(3).sum().reset_index(level=0, drop=True)

In [28]:
print("ESPAÑA")
spain_df = add_rolls(spain_df, SPAIN_FEATURE_REG, ["curated"])

ESPAÑA
curated: añadidas operaciones ['mean', 'max'] sobre ventanas [3, 7, 14] de ['tmax', 'tmin', 'prec', 'racha']


In [29]:
print("CLM")
clm_df = add_rolls(clm_df, CLM_FEATURE_REG, ["curated"])

CLM
curated: añadidas operaciones ['mean', 'max'] sobre ventanas [3, 7, 14] de ['tmax', 'tmin', 'prec', 'racha']


In [30]:
spain_df.loc[
    (spain_df['nombre'] == 'ALBACETE BASE AÉREA') 
    & (spain_df['anio'] == 2015) & (spain_df['mes'].isin([7]))
    & ((spain_df['dia'] <= 29) & (spain_df['dia'] >= 21))
, tmax_target_cols + ["tmax", "tmax_roll_3_max", "tmax_roll_7_max"]]

,fecha,indicativo,tmax,tmax_threshold,extreme_tmax,target_tmax_1,target_tmax_3,target_tmax_7,tmax,tmax_roll_3_max,tmax_roll_7_max
544670,2015-07-21,8175,36.799999,37.959999,0,0,0,0,36.799999,37.000000,37.000000
544671,2015-07-22,8175,36.599998,38.000000,0,0,0,1,36.599998,37.000000,37.000000
544672,2015-07-23,8175,35.299999,38.000000,0,0,0,1,35.299999,36.799999,37.000000
544673,2015-07-24,8175,37.200001,38.000000,0,0,0,1,37.200001,37.200001,37.200001
544674,2015-07-25,8175,34.400002,38.000000,0,0,0,1,34.400002,37.200001,37.200001
544675,2015-07-26,8175,36.900002,38.200001,0,0,1,1,36.900002,37.200001,37.200001
544676,2015-07-27,8175,38.200001,38.200001,0,0,1,1,38.200001,38.200001,38.200001
544677,2015-07-28,8175,37.799999,38.400002,0,1,1,1,37.799999,38.200001,38.200001
544678,2015-07-29,8175,38.799999,38.480000,1,0,0,0,38.799999,38.799999,38.799999


## 4.8. EWMA

In [31]:
# spain_df.groupby("indicativo")["tmax"].ewm(span=3).mean().reset_index(level=0, drop=True)

In [32]:
print("ESPAÑA")
spain_df = add_ewma(spain_df, SPAIN_FEATURE_REG, ["curated"])

ESPAÑA
curated: añadidas EWMA con spans [3, 7, 14] de ['tmax', 'tmin', 'prec', 'racha']


In [33]:
print("CLM")
clm_df = add_ewma(clm_df, CLM_FEATURE_REG, ["curated"])

CLM
curated: añadidas EWMA con spans [3, 7, 14] de ['tmax', 'tmin', 'prec', 'racha']


In [34]:
spain_df.loc[
    (spain_df['nombre'] == 'ALBACETE BASE AÉREA') 
    & (spain_df['anio'] == 2015) & (spain_df['mes'].isin([7]))
    & ((spain_df['dia'] <= 29) & (spain_df['dia'] >= 21))
, tmax_target_cols + ["tmax", "tmax_ewma_7", "tmax_roll_7_mean"]]

,fecha,indicativo,tmax,tmax_threshold,extreme_tmax,target_tmax_1,target_tmax_3,target_tmax_7,tmax,tmax_ewma_7,tmax_roll_7_mean
544670,2015-07-21,8175,36.799999,37.959999,0,0,0,0,36.799999,36.435487,36.157143
544671,2015-07-22,8175,36.599998,38.000000,0,0,0,1,36.599998,36.476615,36.100000
544672,2015-07-23,8175,35.299999,38.000000,0,0,0,1,35.299999,36.182461,35.899999
544673,2015-07-24,8175,37.200001,38.000000,0,0,0,1,37.200001,36.436846,35.957142
544674,2015-07-25,8175,34.400002,38.000000,0,0,0,1,34.400002,35.927635,36.014285
544675,2015-07-26,8175,36.900002,38.200001,0,0,1,1,36.900002,36.170726,36.314286
544676,2015-07-27,8175,38.200001,38.200001,0,0,1,1,38.200001,36.678045,36.485715
544677,2015-07-28,8175,37.799999,38.400002,0,1,1,1,37.799999,36.958534,36.628572
544678,2015-07-29,8175,38.799999,38.480000,1,0,0,0,38.799999,37.418900,36.942857


## 4.9. Diffs

In [35]:
print("ESPAÑA")
spain_df = add_diffs(spain_df, SPAIN_FEATURE_REG, ["curated"])

ESPAÑA
curated: añadidas diferencias con spans [1, 3] de ['mslp_max', 'mslp_min', 'tmax', 'tmin', 'prec', 'racha']


In [36]:
print("CLM")
clm_df = add_diffs(clm_df, CLM_FEATURE_REG, ["curated"])

CLM
curated: añadidas diferencias con spans [1, 3] de ['mslp_max', 'mslp_min', 'tmax', 'tmin', 'prec', 'racha']


In [37]:
# import seaborn as sns
# import matplotlib.pyplot as plt
# sns.kdeplot(spain_df[spain_df["split"]=="train"].groupby("diaAnio")["tmax_diff_1"].mean())
# plt.show()
spain_df.loc[
    (spain_df['nombre'] == 'ALBACETE BASE AÉREA') 
    & (spain_df['anio'] == 2016) & (spain_df['mes'] == 8)
    & (spain_df['dia'] <= 28) & (spain_df['dia'] >= 20)
, tmax_target_cols + ["tmax_diff_1"]]

,fecha,indicativo,tmax,tmax_threshold,extreme_tmax,target_tmax_1,target_tmax_3,target_tmax_7,tmax_diff_1
545066,2016-08-20,8175,34.799999,37.400002,0,0,0,0,-0.200001
545067,2016-08-21,8175,31.500000,37.480000,0,0,0,1,-3.299999
545068,2016-08-22,8175,32.200001,37.480000,0,0,0,1,0.700001
545069,2016-08-23,8175,34.200001,37.400002,0,0,0,1,2.000000
545070,2016-08-24,8175,31.799999,37.380001,0,0,0,1,-2.400002
545071,2016-08-25,8175,32.400002,37.380001,0,0,1,1,0.600002
545072,2016-08-26,8175,33.000000,37.299999,0,0,1,1,0.599998
545073,2016-08-27,8175,35.200001,37.000000,0,1,1,1,2.200001
545074,2016-08-28,8175,37.299999,36.000000,1,0,0,1,2.099998


## 4.10. Historial de extremos

In [38]:
print("ESPAÑA")
spain_df = add_ex_history(spain_df, SPAIN_FEATURE_REG, ["curated"])

ESPAÑA
curated: añadidos historiales de extremos de ['tmax', 'tmin', 'prec', 'racha']


In [39]:
print("CLM")
clm_df = add_ex_history(clm_df, CLM_FEATURE_REG, ["curated"])

CLM
curated: añadidos historiales de extremos de ['tmax', 'tmin', 'prec', 'racha']


In [40]:
spain_df.loc[
    (spain_df['nombre'] == 'ALBACETE BASE AÉREA') 
    & (spain_df['anio'] == 2016) & (spain_df['mes'] == 8)
    & (spain_df['dia'] <= 29) & (spain_df['dia'] >= 20)
, tmax_target_cols + ["days_since_tmax_ex", "extreme_tmax_lag_1"]]

,fecha,indicativo,tmax,tmax_threshold,extreme_tmax,target_tmax_1,target_tmax_3,target_tmax_7,days_since_tmax_ex,extreme_tmax_lag_1
545066,2016-08-20,8175,34.799999,37.400002,0,0,0,0,72.0,0.0
545067,2016-08-21,8175,31.500000,37.480000,0,0,0,1,73.0,0.0
545068,2016-08-22,8175,32.200001,37.480000,0,0,0,1,74.0,0.0
545069,2016-08-23,8175,34.200001,37.400002,0,0,0,1,75.0,0.0
545070,2016-08-24,8175,31.799999,37.380001,0,0,0,1,76.0,0.0
545071,2016-08-25,8175,32.400002,37.380001,0,0,1,1,77.0,0.0
545072,2016-08-26,8175,33.000000,37.299999,0,0,1,1,78.0,0.0
545073,2016-08-27,8175,35.200001,37.000000,0,1,1,1,79.0,0.0
545074,2016-08-28,8175,37.299999,36.000000,1,0,0,1,0.0,0.0
545075,2016-08-29,8175,35.400002,35.919998,0,0,0,1,1.0,1.0


## 4.11. Variables de precipitación

In [41]:
print("ESPAÑA")
spain_df = add_prec_vars(spain_df, SPAIN_FEATURE_REG, ["curated"])

ESPAÑA
curated: añadidas variables de precipitación


In [42]:
print("CLM")
clm_df = add_prec_vars(clm_df, CLM_FEATURE_REG, ["curated"])

CLM
curated: añadidas variables de precipitación


In [43]:
spain_df.loc[
    (spain_df['nombre'] == 'ALBACETE BASE AÉREA') 
    & (spain_df['anio'] == 2016) & (spain_df['mes'] == 8)
    & (spain_df['dia'] <= 25) & (spain_df['dia'] >= 10)
, ["prec", "prec_threshold", "extreme_prec"] + ["prec_roll_7_sum", "days_since_prec"]]

,prec,prec_threshold,extreme_prec,prec_roll_7_sum,days_since_prec
545056,0.5,0.850,0,3.7,0.0
545057,0.0,0.290,0,3.7,1.0
545058,0.0,0.000,0,3.7,2.0
545059,0.0,0.000,0,3.7,3.0
545060,0.0,0.250,0,3.7,4.0
545061,NaN,2.215,0,NaN,0.0
545062,1.9,1.835,1,NaN,0.0
545063,0.0,2.850,0,NaN,1.0
545064,0.0,2.830,0,NaN,2.0
545065,0.0,3.600,0,NaN,3.0


## 4.12. Características auxiliares

In [44]:
print("ESPAÑA")
spain_df = add_aux(spain_df, SPAIN_FEATURE_REG, ["minimal", "curated"])

ESPAÑA
minimal: añadidas variables auxiliares
curated: añadidas variables auxiliares


In [45]:
print("CLM")
clm_df = add_aux(clm_df, CLM_FEATURE_REG, ["minimal", "curated"])

CLM
minimal: añadidas variables auxiliares
curated: añadidas variables auxiliares


# 5. Características

In [46]:
spain_feats = pd.DataFrame(SPAIN_FEATURE_REG).sort_values(["view", "var"]).reset_index(drop=True)
print(spain_feats.head())

                  var     view
0             altitud  curated
1                anio  curated
2             cos_doy  curated
3     days_since_prec  curated
4  days_since_prec_ex  curated


In [47]:
clm_feats = pd.DataFrame(CLM_FEATURE_REG).sort_values(["view", "var"]).reset_index(drop=True)

# 6. Guardado

In [48]:
from features import save_data

In [49]:
save_data(spain_df, CONFIG["out_features_spain"], spain_feats, CONFIG["reg_path_spain"])

Datos guardados en data/processed/features_df_spain.parquet
    Tamaño de datos: (799338, 132)
Registro de características guardado en data/processed/feature_reg_spain.csv
    Longitud vista completa: 100
    Longitud vista mínima: 28


In [50]:
save_data(clm_df, CONFIG["out_features_clm"], clm_feats, CONFIG["reg_path_clm"])

Datos guardados en data/processed/features_df_clm.parquet
    Tamaño de datos: (339698, 130)
Registro de características guardado en data/processed/feature_reg_clm.csv
    Longitud vista completa: 100
    Longitud vista mínima: 28
